In [59]:
import geemap
import ee
import pandas as pd
import geopandas as gpd
import rasterio as rio

# Authenticate once in this machine/session before running the notebook end-to-end.
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com',
)

# Set the path to the JSON file containing the geometry.
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
gdf_menor = gdf[gdf['nm_mesoRH'] == 'Baixo São Francisco']
geom_ee = geemap.geopandas_to_ee(gdf_menor)
area = geom_ee.geometry()

# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2000-02-18', '2025-12-31')

In [60]:
# 4. Função para calcular MSAVI e Albedo
def add_indices(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real [7]
    img_scaled = image.multiply(0.0001)

    # Cálculo do MSAVI usando .expression() [4]
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': img_scaled.select('sur_refl_b02'), # Banda NIR no MODIS
            'RED': img_scaled.select('sur_refl_b01')  # Banda RED no MODIS
        }).rename('MSAVI')

    # Cálculo do Albedo (Exemplo usando fórmula empírica comum para MODIS)
    # Verifique os coeficientes exatos da metodologia que você está seguindo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015', {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }).rename('Albedo')

    # Adiciona as novas bandas à imagem original e mantém as propriedades de data [4, 8]
    return image.addBands([msavi, albedo]).copyProperties(image, ['system:time_start'])

In [67]:
# 5. Mapear a função sobre toda a coleção de imagens
modis_processado = modis.map(add_indices)

# Testar a performance no XEE

In [73]:
# 1. Inicializar o XEE
import xarray as xr
from xee import helpers
import geopandas as gpd


aoi = gdf_menor.geometry.union_all()

# Definir parâmetros para fit da geometria
CRS = 'EPSG:4326'
GRID_SCALE = (0.005, -0.005)

grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=CRS,
    grid_crs=CRS,
    grid_scale=GRID_SCALE,
)

# Filtrar a coleção apenas com as bandas de interesse (MSAVI e Albedo)
modis_processado = modis_processado.select(['MSAVI', 'Albedo'])

# Abre a coleção do GEE como um cubo de dados multidimensional
ds = xr.open_dataset(
    modis_processado,
    engine='ee',
    **grid_params,
    chunks='auto',
)

# ds = ds.mean(dim='time') * 1


## ABORDAGEM 1: DDI (Desertification Divided Index) - Regressão Linear
- Utilizando o coeficiente empírico (K = 1.803) validado na literatura [11]
- Valores MAIORES indicam áreas severamente preservadas (ou não-desertificadas, dependendo do sinal do índice).
- A literatura em [11] frequentemente inverte o eixo para mapear Intensidade I.
- No modelo padrão DDI (K * MSAVI - Albedo), o decréscimo representa degradação [10].

## ABORDAGEM 2: SASDI (Point-to-Point Distance Model)
- Calcula a distância euclidiana do pixel em relação ao estado ideal (MSAVI=1, Albedo=0) [12, 13]
- Quanto MAIOR a distância, MAIOR o grau de desertificação.
- O xarray aplica a operação vetorial elemento a elemento perfeitamente no Dask Graph.
- ---------------------------------------------------------

In [78]:
import rasterio as rio
import rioxarray as rioxr
import numpy as np
from scipy import stats

In [ ]:
# 1. Criar a função estatística que será aplicada em cada pixel
def regressao_pixel(x, y):
    # x = MSAVI_max, y = Albedo_min (vetores 1D ao longo do tempo)
    # Máscara para pular os anos que tiverem NaN no pixel
    mask = ~np.isnan(x) & ~np.isnan(y)
    valid_x = x[mask]
    valid_y = y[mask]
    
    # Exige pelo menos 3 anos de dados válidos para calcular a regressão
    if len(valid_x) > 2:
        # Calcula a regressão linear: Y (Albedo) em função de X (MSAVI)
        slope, intercept, r_value, p_value, std_err = stats.linregress(valid_x, valid_y)
        r2 = r_value**2
        return slope, r2, p_value
    else:
        return np.nan, np.nan, np.nan

In [85]:
# 1. Agrupar por ano e calcular os extremos anuais
msavi_max_anual = ds['MSAVI'].groupby('time.year').max(dim='time')
albedo_min_anual = ds['Albedo'].groupby('time.year').min(dim='time')

# 2. Criar um novo Dataset limpo apenas com os dados anuais
ds_anual = xr.Dataset({
    'MSAVI_max': msavi_max_anual,
    'Albedo_min': albedo_min_anual
}).chunk({'year': -1})

# 1. Calcular mínimos e máximos globais da série para a normalização
# Isso garante que a escala de 0 a 1 seja consistente ao longo de todos os anos
M_min = ds_anual['MSAVI_max'].min()
M_max = ds_anual['MSAVI_max'].max()

A_min = ds_anual['Albedo_min'].min()
A_max = ds_anual['Albedo_min'].max()

# 2. Aplicar a normalização para criar as variáveis M e A (escaladas de 0 a 1)
ds_anual['M_norm'] = (ds_anual['MSAVI_max'] - M_min) / (M_max - M_min)
ds_anual['A_norm'] = (ds_anual['Albedo_min'] - A_min) / (A_max - A_min)

# 3. Aplicar a função no Dataset anual usando apply_ufunc
# Isso iterará a função acima sobre o eixo do tempo (year) para cada pixel (y, x)
k_slope, r2, p_value = xr.apply_ufunc(
    regressao_pixel,
    ds_anual['M_norm'],
    ds_anual['A_norm'],
    input_core_dims=[['year'], ['year']], # Dimensão que será "consumida" pela regressão
    output_core_dims=[[], [], []],        # Retorna 3 valores escalares por pixel (sem a dimensão ano)
    vectorize=True,                       # Permite que a função lide com os arrays do xarray/dask
    dask='parallelized',                  # Habilita o processamento via Dask
    output_dtypes=[float, float, float]
)

# 4. Consolidar os resultados em um novo Dataset
ds_estatisticas = xr.Dataset({
    'k_slope': k_slope,
    'r2': r2,
    'p_value': p_value
})

# 5. Calcular o coeficiente alpha (α = 1 / K)
# O xarray permite operações matemáticas diretas mantendo a estrutura espacial
ds_estatisticas['alpha'] = 1 / ds_estatisticas['k_slope']

In [86]:
# 3. Cálculo do DDI
# DDI = k * M - A 
# (Utilizamos o k_slope que calculamos via regressão espacial com o apply_ufunc)
ds_anual['DDI'] = (ds_estatisticas['k_slope'] * ds_anual['M_norm']) - ds_anual['A_norm']

# 4. Cálculo do SASDI
# SASDI = raiz((M - 1)² + A²)
ds_anual['SASDI'] = np.sqrt((ds_anual['M_norm'] - 1)**2 + ds_anual['A_norm']**2)

In [33]:
import time
from tqdm import tqdm

In [88]:
with tqdm(total=100) as pbar:
    # 1. Junta as estatísticas ao dataset principal para ter tudo em um só arquivo
    ds_final = xr.merge([ds_anual, ds_estatisticas])
    # 2. Computa todas as operações (isso pode levar alguns minutos dependendo do tamanho da área)
    ds_final = ds_final.compute()
    # 3. Salva no disco
    ds_final.to_netcdf('../data/indices_desertificacao_anual.nc')
    pbar.update(100)
    print("Processamento concluído e salvo com sucesso!")

100%|██████████| 100/100 [05:33<00:00,  3.34s/it]

Processamento concluído e salvo com sucesso!


### Visualização da Série Temporal dos Índices de Desertificação

Vamos plotar as séries temporais do DDI e SASDI para observar as tendências ao longo do tempo.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, ax = plt.subplots(figsize=(10, 4), subplot_kw={'projection': ccrs.PlateCarree()})

# 1. Plotar a geometria da bacia
ax.add_geometries(gdf_menor.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=1)

# 2. Definir o zoom na bacia
bounds = gdf_menor.total_bounds # [minx, miny, maxx, maxy]
ax.set_extent([bounds[0], bounds[2], bounds[1], bounds[3]], crs=ccrs.PlateCarree())

# 3. Plotar o DDI com imshow rápido
# ATENÇÃO: Se o seu DataArray tiver dimensão 'time', selecione um tempo único com .isel(time=0)
ddi_plot = ds['DDI'].plot.imshow(
    ax=ax,
    transform=ccrs.EqualEarth(), # Ou ccrs.PlateCarree(), veja os detalhes abaixo!
    cmap='viridis',
    alpha=0.6,
    add_colorbar=True,
    robust=True # Ajusta o contraste ignorando outliers
)